## \-\-- Day 7: Some Assembly Required \-\--

This year, Santa brought little Bobby Tables a set of wires and [bitwise logic gates](https://en.wikipedia.org/wiki/Bitwise_operation)! Unfortunately, little Bobby is a little under the recommended age range, and he needs help assembling the circuit.

Each wire has an identifier (some lowercase letters) and can carry a [16-bit](https://en.wikipedia.org/wiki/16-bit) signal (a number from `0` to `65535`). A signal is provided to each wire by a gate, another wire, or some specific value. Each wire can only get a signal from one source, but can provide its signal to multiple destinations. A gate provides no signal until all of its inputs have a signal.

The included instructions booklet describes how to connect the parts together: `x AND y -> z` means to connect wires `x` and `y` to an AND gate, and then connect its output to wire `z`.

For example:

-   `123 -> x` means that the signal `123` is provided to wire `x`.

-   `x AND y -> z` means that the [bitwise AND](https://en.wikipedia.org/wiki/Bitwise_operation#AND) of wire `x` and wire `y` is provided to wire `z`.

-   `p LSHIFT 2 -> q` means that the value from wire `p` is [left-shifted](https://en.wikipedia.org/wiki/Logical_shift) by `2` and then provided to wire `q`.

-   `NOT e -> f` means that the [bitwise complement](https://en.wikipedia.org/wiki/Bitwise_operation#NOT) of the value from wire `e` is provided to wire `f`.

Other possible gates include `OR` ([bitwise OR](https://en.wikipedia.org/wiki/Bitwise_operation#OR)) and `RSHIFT` ([right-shift](https://en.wikipedia.org/wiki/Logical_shift)). If, for some reason, you'd like to *emulate* the circuit instead, almost all programming languages (for example, [C](https://en.wikipedia.org/wiki/Bitwise_operations_in_C), [JavaScript](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Operators/Bitwise_Operators), or [Python](https://wiki.python.org/moin/BitwiseOperators)) provide operators for these gates.

For example, here is a simple circuit:

    123 -> x
    456 -> y
    x AND y -> d
    x OR y -> e
    x LSHIFT 2 -> f
    y RSHIFT 2 -> g
    NOT x -> h
    NOT y -> i

After it is run, these are the signals on the wires:

    d: 72
    e: 507
    f: 492
    g: 114
    h: 65412
    i: 65079
    x: 123
    y: 456

In little Bobby's kit's instructions booklet (provided as your puzzle input), what signal is ultimately provided to *wire `a`*?


In [ ]:
with open('data-2015-7.txt', 'r') as f:
    logics = f.read().splitlines()

This might be a similar construction to yesterday's instructions:

-   input
-   wire1
-   operation
-   wire2
-   output

Sometimes there is only input and output i.e. if it's a number on the left hand side. For a `NOT` operator there is only wire2.

Operators are:

-   `NOT ~`
-   `AND &`
-   `OR |`
-   `RSHIFT >>`
-   `LSHIFT <<`


In [ ]:
print('Total: ' + str(len(logics)))

def lens(lst, op='AND'):
    return len([x for x in lst if op in x])

print('AND: ' + str(lens(logics, 'AND')))  
print('OR: ' + str(lens(logics, 'OR')))  
print('NOT: ' + str(lens(logics, 'NOT')))  
print('RSHIFT: ' + str(lens(logics, 'RSHIFT')))  
print('LSHIFT: ' + str(lens(logics, 'LSHIFT')))  

print('Sub-Total:' +  
      str(lens(logics, 'AND') + lens(logics, 'OR') + 
          lens(logics, 'NOT') + lens(logics, 'RSHIFT') + 
          lens(logics, 'LSHIFT')))

The remaining instructions after all the others above:

-   `1674 -> b`
-   `0 -> c`
-   `lx -> a`

I'm not sure how to make this efficient. There doesn't seem like a good way to loop through these instructions to fill out wires and signals. We need to gather a dictionary of all the possible wires first, I think, then populate that dictionary.


In [ ]:
# get the before and after pieces
logic_split = [x.split(' -> ') for x in logics]

# we want to get the targets in a dictionary and then populate that dictionary
targets_list = [x[1] for x in logic_split]
targets = dict.fromkeys(targets_list)

# split by space for signals
signals_list = [x[0].split(' ') + [False] for x in logic_split]

I think we're in a good place to start iterating.


In [ ]:
i = 0

while None in targets.values():
# while i < 100:  
    # sum([1 for x in targets.values() if x is None])
    i += 1
    for s,t in zip(signals_list, targets_list):
        if s[-1]:
            # we've already processed
            if targets[t] is None: 
                raise('This is awful')
        elif len(s) == 4:
            # get some information from the signals
            try: m = targets[s[0]]
            except: m = int(s[0])
            try: n = targets[s[2]]
            except: n = int(s[2])
            
            # these need to both be populated
            if isinstance(m, int) and isinstance(n, int):
                print(str(m) + ' ' + s[1] + ' ' + str(n) + ' -> ' + t)
                if s[1] == 'AND':
                    targets[t] = m & n
                elif s[1] == 'OR':
                    targets[t] = m | n
                elif s[1] == 'LSHIFT':
                    targets[t] = m << n
                elif s[1] == 'RSHIFT':
                    targets[t] = m >> n
                else: 
                    raise('this is bad')
                # skip this instruction next time
                s[-1] = True
        # this is always NOT
        elif len(s) == 3:
            if isinstance(targets[s[1]], int):
                print('NOT ' + str(s[1]) + ' -> ' + t)
                targets[t] = ~ targets[s[1]]
                s[-1] = True
        # this is usually a number, but might also be the answer
        elif len(s) == 2:
            if s[0].isdigit():
                print('ONE: ' + str(s[0]) + ' -> ' + t)
                targets[t] = int(s[0])
                s[-1] = True
            else: 
                # print(t + ': ' + str(s[0]))
                targets[t] = targets[s[0]]
        else:
            raise('this is not good')


## \-\-- Part Two \-\--

Now, take the signal you got on wire `a`, override wire `b` to that signal, and reset the other wires (including wire `a`). What new signal is ultimately provided to wire `a`?
